<a href="https://colab.research.google.com/github/Sharmadipti/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sharmadipti/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### My unit of analysis and time window

One row represents one content page for one client over a monthly observation period.

I will use February 2026 as the feature window. These features represent information that would have been available by the end of February.

I will use March 2026 as the outcome window. The March outcome will be used as the label for what happened after the feature period.

Keeping the feature and outcome windows separate helps prevent future information from leaking into the features.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Fields: feature / label / context / excluded

**Features:** February impressions, clicks, CTR, average position, and content age. These are used because they are measurable before the March outcome.

**Label:** A March outcome indicating whether a page lost click visibility. This is a provisional target for identifying pages that may deserve review.

**Context:** Content type, search intent, word count, and freshness information can help explain differences between pages.

**Excluded:** Client names, URLs, private search queries, and any information from after the decision point are excluded because they are not needed for the decision and could cause leakage or expose sensitive information.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import getpass
import duckdb

def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token

    return getpass.getpass("Paste your Hugging Face READ token: ")

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN '{get_hf_token()}')
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Connected to the FlyRank warehouse.")

query_1 = f"""
SELECT
    client_hash_id,
    content_hash_id,
    COUNT(*) AS daily_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {MAR}
GROUP BY client_hash_id, content_hash_id
ORDER BY daily_rows DESC
LIMIT 10
"""

grain_check = con.sql(query_1).df()

grain_check

query_2 = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {MAR}
"""

slice_check = con.sql(query_2).df()

print("March 2026 slice:")
slice_check

query_3 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_with_gsc_data
FROM {MAR}
"""

availability_check = con.sql(query_3).df()

print("GSC data availability for March 2026:")
availability_check

# ---------------------------------------------------------
# Five-feature frame
# February = feature window
# March = label window
# ---------------------------------------------------------

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
DIM = f"read_parquet('{REL}/dim_content.parquet')"

feature_query = f"""
WITH feb_features AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_feb,
        SUM(gsc_clicks) AS clicks_feb,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr_feb,
        SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position_feb
    FROM {FEB}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

march_label AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS clicks_mar
    FROM {MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.impressions_feb,
    f.clicks_feb,
    f.ctr_feb,
    f.avg_position_feb,
    DATE_DIFF(
        'day',
        c.content_created_date,
        DATE '2026-02-28'
    ) AS content_age_days,
    CASE
        WHEN COALESCE(m.clicks_mar, 0) = 0 THEN 1
        ELSE 0
    END AS went_dark
FROM feb_features f
LEFT JOIN march_label m
    ON f.client_hash_id = m.client_hash_id
   AND f.content_hash_id = m.content_hash_id
LEFT JOIN {DIM} c
    ON f.content_hash_id = c.content_hash_id
WHERE f.impressions_feb >= 100
  AND f.clicks_feb >= 3
  AND c.is_published IS TRUE
  AND c.content_created_date <= DATE '2026-02-28'
"""

feature_frame = con.sql(feature_query).df()

print(f"Feature-frame rows: {len(feature_frame):,}")
print("\nFive features:")
print([
    "impressions_feb",
    "clicks_feb",
    "ctr_feb",
    "avg_position_feb",
    "content_age_days"
])

feature_frame[
    [
        "client_hash_id",
        "content_hash_id",
        "impressions_feb",
        "clicks_feb",
        "ctr_feb",
        "avg_position_feb",
        "content_age_days",
        "went_dark"
    ]
].head()


Paste your Hugging Face READ token: ··········
Connected to the FlyRank warehouse.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March 2026 slice:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

GSC data availability for March 2026:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-frame rows: 29,700

Five features:
['impressions_feb', 'clicks_feb', 'ctr_feb', 'avg_position_feb', 'content_age_days']


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,ctr_feb,avg_position_feb,content_age_days,went_dark
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,601.0,3.0,0.004992,4.633943,365,0
1,client_73cda7b4e4f265ea,content_05434271b257bb68,1567.0,7.0,0.004467,5.641353,365,0
2,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2176.0,4.0,0.001838,3.807445,365,0
3,client_73cda7b4e4f265ea,content_712c365258cee05c,5368.0,21.0,0.003912,3.818927,365,0
4,client_73cda7b4e4f265ea,content_8935ed68eca88b01,3248.0,7.0,0.002155,10.112069,365,0


### Five features and when they are available

**1. impressions_feb** — Knowable at the decision moment because these impressions were recorded during the February feature window.

**2. clicks_feb** — Knowable at the decision moment because these clicks were recorded during the February feature window.

**3. ctr_feb** — Knowable at the decision moment because it is calculated from February impressions and clicks only.

**4. avg_position_feb** — Knowable at the decision moment because it is calculated from search-position information available during February.

**5. content_age_days** — Knowable at the decision moment because it is calculated from the content creation date as of the end of February.

In [14]:
# ---------------------------------------------------------
# Leakage trap: deliberately include the label as a feature
# ---------------------------------------------------------

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

five_features = [
    "impressions_feb",
    "clicks_feb",
    "ctr_feb",
    "avg_position_feb",
    "content_age_days"
]

X = feature_frame[five_features].copy()
y = feature_frame["went_dark"].astype(int)

# Handle any missing values
X = X.fillna(X.median())

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

honest_predictions = model.predict(X_test)
honest_score = accuracy_score(y_test, honest_predictions)

print(f"Honest accuracy using five features: {honest_score:.3f}")


# ---------------------------------------------------------
# Deliberate leakage experiment
# Add the label itself as a feature on purpose
# ---------------------------------------------------------

X_leak = feature_frame[five_features + ["went_dark"]].copy()
X_leak = X_leak.fillna(X_leak.median())

X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_leak,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

leak_model = LogisticRegression(max_iter=1000)
leak_model.fit(X_train_leak, y_train_leak)

leak_predictions = leak_model.predict(X_test_leak)
leak_score = accuracy_score(y_test_leak, leak_predictions)

print(f"Leaky accuracy with label-derived feature: {leak_score:.3f}")

print("\nLeakage feature used deliberately: went_dark")
print("The leaky feature is the label itself, so this score is NOT valid.")


# ---------------------------------------------------------
# Remove the leakage feature
# Keep only the five legitimate features
# ---------------------------------------------------------

print("\nFinal honest feature set:")
print(five_features)

print(f"Final honest accuracy to keep: {honest_score:.3f}")

Honest accuracy using five features: 0.949
Leaky accuracy with label-derived feature: 1.000

Leakage feature used deliberately: went_dark
The leaky feature is the label itself, so this score is NOT valid.

Final honest feature set:
['impressions_feb', 'clicks_feb', 'ctr_feb', 'avg_position_feb', 'content_age_days']
Final honest accuracy to keep: 0.949


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limitation

One important limitation is that GSC data is not available for every warehouse row. In the March 2026 slice, only 3,611,061 of 9,841,378 rows had GSC data available. Therefore, pages with missing GSC data may be underrepresented in features based on search performance.

The March outcome window also has to be treated carefully because it is the future outcome relative to the February feature window. The final month should not be used to develop the label logic because it represents the natural outcome window for earlier observations.

Therefore, the results should be treated as decision-support for the available slice rather than as a complete picture of every page or client.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.